# Robust galaxy morphology across environment-selected samples

This notebook turns the action items from the 13 July 2026 research meeting into a reproducible analysis. It compares robust early-type and late-type galaxy classifications in the `highlum` and `highdens` cross-match products.

The notebook is intentionally explanatory: every figure is preceded by its question and construction, then followed by guidance on interpretation and limitations. Generated outputs are local artifacts and are not committed.

In [ ]:
# Centralize every reproducibility and output-schema setting so one cell documents the full run contract.
import json
import os
import platform
from pathlib import Path

import numpy as np

SEED = 20260713
MAX_SEPARATION_ARCSEC = 1.0
FLUX_RADIUS_CUT = 50.0
PLOT_SAMPLE_SIZE = 100_000
# Fixed limits make the saved robust-class panels directly comparable across catalogues.
COMPARISON_MAGNITUDE_LIMITS = (21.6, 18.0)
COMPARISON_RADIUS_LIMITS = (2.5, 22.0)
# Exact headers are executable contracts: the notebook reads generated CSVs back and fails if a name drifts.
QUALITY_HEADERS = ('catalog', 'column', 'dtype', 'unit', 'n_total', 'n_finite', 'n_missing', 'n_out_of_range', 'min', 'max', 'mean', 'median')
FLAG_COUNT_HEADERS = ('catalog', 'flag_ltg', 'class_label', 'count', 'fraction_of_total', 'wilson_low_95', 'wilson_high_95')
SUMMARY_HEADERS = ('catalog', 'class', 'variable', 'n_total', 'n_valid', 'n_missing', 'mean', 'median', 'std_ddof1', 'min', 'p05', 'p25', 'p50', 'p75', 'p95', 'max', 'iqr')
THRESHOLD_HEADERS = ('catalog', 'threshold', 'n_valid', 'n_above', 'fraction_above', 'n_below', 'fraction_below', 'n_flag5', 'true_positive', 'true_negative', 'false_positive', 'false_negative', 'agreement_with_flag5')
FILTER_AUDIT_HEADERS = ('catalog', 'stage', 'rule', 'n_before', 'n_removed', 'n_after', 'fraction_removed')

def find_project_root() -> Path:
    """Locate the repository from an explicit environment value or a notebook working directory.

    This makes the notebook work from Jupyter, nbconvert, the main checkout, or an isolated worktree
    without embedding Mayra's absolute machine path in committed code.
    """
    configured = os.environ.get('GALAXY_PROJECT_ROOT')
    candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'data' / 'README.md').is_file():
            return candidate.resolve()
    raise FileNotFoundError('project root requires requirements.txt and data/README.md')

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'meeting-2026-07-13'


## 1. Scientific context and questions

Modern surveys contain far more galaxy images than a research team can inspect consistently by hand. The Vega-Ferrero catalogue applies ensembles of convolutional neural networks to DES images and reports probabilities and flags for two related questions: whether a galaxy has late-type morphology and whether a disk is viewed edge-on.

In this notebook, **early-type galaxy (ETG)** and **late-type galaxy (LTG)** are operational catalogue classes, not claims that every galaxy belongs to a perfect physical binary. We ask: (1) how many robust ETGs and LTGs occur in each cross-match, (2) how their classification probabilities and observable size/brightness measures differ, and (3) whether patterns differ between the files called `highlum` and `highdens`.

The environmental catalogues have their own selection functions. A different ETG/LTG fraction can therefore arise from population differences, survey limits, matching, or classification uncertainty. The analysis describes associations; it does not establish environmental causation.

## 2. Data inventory and provenance

Five local FITS files have distinct roles. `DES_DR1_CNN_morphological_catalog.fit` is the 26.97-million-row parent morphology catalogue. The two `redspell_*` files are environmental source catalogues. The two `match_VF_*` files are derived coordinate matches that append Vega-Ferrero morphology fields and a match separation to the environmental records.

Only the parent morphology catalogue and the two match products are analyzed directly here. Source catalogues are inspected for provenance. Raw inputs, derived match products, and generated results remain in separate directories so an analysis cannot silently overwrite its evidence.

In [ ]:
# Make the repository package importable even when Jupyter starts inside the notebooks/ directory.
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from astropy.table import Table
from src.galaxy_analysis.catalog import inspect_catalog, random_indices, read_columns

# One alias-to-path registry prevents later cells from silently opening a different catalogue.
CATALOG_PATHS = {
    'parent_morphology': PROJECT_ROOT / 'data' / 'DES_DR1_CNN_morphological_catalog.fit',
    'highlum_source': PROJECT_ROOT / 'data' / 'raw' / 'red-sequence' / 'redspell_highlum7_final.fits',
    'highdens_source': PROJECT_ROOT / 'data' / 'raw' / 'red-sequence' / 'redspell_highdens7_final.fits',
    'highlum': PROJECT_ROOT / 'data' / 'processed' / 'crossmatches' / 'vega-ferrero' / 'match_VF_highlum.fits',
    'highdens': PROJECT_ROOT / 'data' / 'processed' / 'crossmatches' / 'vega-ferrero' / 'match_VF_highdens.fits',
}
# Schema inspection reads headers only; no multi-gigabyte table is copied at this stage.
schemas = {name: inspect_catalog(path) for name, path in CATALOG_PATHS.items()}
inventory = Table(rows=[
    (name, str(schema.path.relative_to(PROJECT_ROOT)), schema.row_count, schema.column_count, schema.hdu_index, schema.extname)
    for name, schema in schemas.items()
], names=('catalog', 'path', 'rows', 'columns', 'table_hdu', 'extname'))
inventory


## 3. Reproducibility configuration

Every random operation uses seed `20260713`, the meeting date. Coordinate matches are accepted only through 1.0 arcsec. The proposed `FLUX_RADIUS_R < 50` cut is used only for an explicitly labeled diagnostic figure until its meaning is confirmed. Scatter rendering is capped at 100,000 deterministic points, while numerical statistics use all valid rows.

The notebook writes tables, figures, and run metadata below `outputs/meeting-2026-07-13/`. That directory is ignored by Git: generated outputs can always be rebuilt and must not be confused with source code or input data.

## 4. FITS schema and quality validation

A catalogue can be opened successfully and still be unsuitable for analysis. We therefore check structure, domains, missing values, duplicate identifiers, and coordinate-match quality before calculating any scientific summary. The match tolerance is an **angular separation** on the sky. One arcsecond is not a physical distance: its physical scale depends on redshift and cosmology.

`Separation <= 1 arcsec` follows the Topcat procedure demonstrated in the meeting. Invalid rows are counted rather than silently discarded. Missing coordinate units in a FITS header are reported as metadata limitations even when the catalogue convention indicates degrees.

In [ ]:
# Validate the smaller highlum product first; its verified baseline is the pipeline regression checkpoint.
from dataclasses import asdict

from src.galaxy_analysis.selection import audit_filter, robust_masks, valid_value_mask, validate_separation

# Request only the 21 columns used below instead of materializing all 191 FITS columns.
CORE_COLUMNS = (
    'COADD_OBJECT_ID', 'object_id', 'RA_2', 'DEC_2', 'MAG_AUTO_R',
    'FLUX_RADIUS_R', 'MP_LTG', 'MP_EdgeOn', 'FLAG_LTG', 'FLAG_EdgeOn',
    'Separation', 'P1_LTG', 'P2_LTG', 'P3_LTG', 'P4_LTG', 'P5_LTG',
    'P1_EdgeOn', 'P2_EdgeOn', 'P3_EdgeOn', 'P4_EdgeOn', 'P5_EdgeOn',
)
highlum = read_columns(CATALOG_PATHS['highlum'], CORE_COLUMNS)
# Keep one named mask per scientific rule so every removal can be audited and written to CSV.
quality_masks = {
    'RA_2 in [0, 360) deg': valid_value_mask(highlum['RA_2'], 'ra_deg'),
    'DEC_2 in [-90, 90] deg': valid_value_mask(highlum['DEC_2'], 'dec_deg'),
    'FLUX_RADIUS_R positive': valid_value_mask(highlum['FLUX_RADIUS_R'], 'positive'),
    'MP_LTG in [0, 1]': valid_value_mask(highlum['MP_LTG'], 'probability'),
    'MP_EdgeOn in [0, 1]': valid_value_mask(highlum['MP_EdgeOn'], 'probability'),
    'Separation in [0, 1] arcsec': valid_value_mask(highlum['Separation'], 'nonnegative') & (highlum['Separation'] <= MAX_SEPARATION_ARCSEC),
}
audit_rows = [audit_filter('highlum quality', rule, mask) for rule, mask in quality_masks.items()]
separation_check = validate_separation(highlum['Separation'], MAX_SEPARATION_ARCSEC)
duplicate_morphology_ids = len(highlum['COADD_OBJECT_ID']) - len(np.unique(highlum['COADD_OBJECT_ID']))
duplicate_environment_ids = len(highlum['object_id']) - len(np.unique(highlum['object_id']))
# Match-separation failure is fatal: normal-looking downstream plots would otherwise hide a bad cross-match.
assert separation_check.is_valid, f'{separation_check.invalid_count} matches exceed quality limits'
audit_table = Table(rows=[('highlum', *tuple(asdict(row).values())) for row in audit_rows], names=FILTER_AUDIT_HEADERS)
audit_table


In [ ]:
# Write reproducible diagnostics below ignored outputs/, keeping generated evidence separate from input data.
highlum_output = OUTPUT_ROOT / 'highlum'
highlum_tables = highlum_output / 'tables'
highlum_figures = highlum_output / 'figures'
highlum_tables.mkdir(parents=True, exist_ok=True)
highlum_figures.mkdir(parents=True, exist_ok=True)
# Summarize each loaded column without converting the full 191-column catalogue to an Astropy Table.
quality_rows = []
for name, values in highlum.items():
    array = np.asarray(values)
    if np.issubdtype(array.dtype, np.number):
        numeric = array.astype(float)
        finite = np.isfinite(numeric)
        valid = numeric[finite]
        stats = (float(valid.min()), float(valid.max()), float(valid.mean()), float(np.median(valid))) if valid.size else (None, None, None, None)
        n_finite, n_missing = int(finite.sum()), int((~finite).sum())
    else:
        stats = (None, None, None, None)
        n_finite, n_missing = int(array.size), 0
    quality_rows.append(('highlum', name, str(array.dtype), schemas['highlum'].column_units.get(name), len(array), n_finite, n_missing, 0, *stats))
quality_table = Table(rows=quality_rows, names=QUALITY_HEADERS)
# overwrite=True is safe here because outputs are deterministic run artifacts, never source data.
quality_table.write(highlum_tables / 'catalog_quality.csv', format='ascii.csv', overwrite=True)
quality_report = {
    'catalog': 'highlum',
    'rows': schemas['highlum'].row_count,
    'columns': schemas['highlum'].column_count,
    'separation_unit': schemas['highlum'].column_units['Separation'],
    'separation_invalid_count': separation_check.invalid_count,
    'separation_maximum_valid_arcsec': separation_check.maximum_valid,
    'duplicate_morphology_ids': int(duplicate_morphology_ids),
    'duplicate_environment_ids': int(duplicate_environment_ids),
    'warnings': ['RA/DEC units are not declared in the FITS header; catalogue convention is degrees'],
}
(highlum_output / 'quality_report.json').write_text(json.dumps(quality_report, indent=2), encoding='utf-8')
audit_table.write(highlum_tables / 'filter_audit.csv', format='ascii.csv', overwrite=True)
# Read outputs back immediately so serialization/header drift fails during execution.
assert tuple(Table.read(highlum_tables / 'catalog_quality.csv', format='ascii.csv').colnames) == QUALITY_HEADERS
assert tuple(Table.read(highlum_tables / 'filter_audit.csv', format='ascii.csv').colnames) == FILTER_AUDIT_HEADERS
quality_report


## 5. Robust ETG and LTG selection

The morphology flag encodes both class and confidence tier. The primary analysis uses exactly `FLAG_LTG == 4` for robust ETGs and `FLAG_LTG == 5` for robust LTGs. Flags 0–3 are excluded from the robust comparison. Treating every even flag as ETG and every odd flag as LTG would mix lower-confidence objects into the samples and would not follow the meeting decision.

A small robust-LTG fraction is not, by itself, evidence that the classifier failed. The environmental source catalogue may preferentially select luminous, dense-region, or red-sequence systems, producing **sample-selection bias** toward ETGs. Class imbalance is therefore reported as a result and interpreted cautiously.

In [ ]:
# Apply only the documented robust flags: 4=ETG and 5=LTG; flags 0–3 remain outside the primary sample.
highlum_masks = robust_masks(highlum['FLAG_LTG'])
flag_values, flag_counts = np.unique(highlum['FLAG_LTG'], return_counts=True)
highlum_flag_counts = dict(zip(flag_values.astype(int), flag_counts.astype(int)))
# These exact local baselines catch wrong files, wrong HDUs, or accidental changes to selection logic.
assert len(highlum['FLAG_LTG']) == 34_768
assert highlum_flag_counts == {0: 5220, 1: 3039, 2: 344, 3: 404, 4: 24871, 5: 890}
assert int(highlum_masks.robust_etg.sum()) == 24_871
assert int(highlum_masks.robust_ltg.sum()) == 890
selection_table = Table(
    rows=[('robust ETG', 4, int(highlum_masks.robust_etg.sum())), ('robust LTG', 5, int(highlum_masks.robust_ltg.sum()))],
    names=('class', 'FLAG_LTG', 'count'),
)
selection_table['fraction_of_catalog'] = selection_table['count'] / len(highlum['FLAG_LTG'])
selection_table


## 6. Classification probabilities and model agreement

A `FLAG_LTG` value is a catalogue decision assembled from model outputs and confidence rules. `MP_LTG` and `MP_EdgeOn` are continuous aggregate probabilities, while P1–P5 preserve the five model predictions. These quantities answer different questions: the flag defines the selected class; the aggregate probability shows the strength of the model output; and the spread across P1–P5 shows inter-model disagreement.

We report mean, median, sample standard deviation (`ddof=1`), percentiles, and interquartile range. Thresholds 0.5, 0.6, and 0.8 are sensitivity checks only. Agreement with flag 5 does not prove correctness because the flag is not an independent human-labelled ground truth. High confidence can still be systematically wrong.

In [ ]:
# Compute numerical summaries from every valid row; rendering samples are introduced only in later plot cells.
from src.galaxy_analysis.statistics import describe_values, flag_count_rows, model_dispersion, threshold_rows

highlum_flag_rows = flag_count_rows('highlum', highlum['FLAG_LTG'])
flag_table = Table(rows=[tuple(asdict(row).values()) for row in highlum_flag_rows], names=FLAG_COUNT_HEADERS)
# Use the same variables and definitions for the complete sample and both robust classes.
summary_rows = []
for class_name, mask in [('all_valid', np.ones(len(highlum['FLAG_LTG']), dtype=bool)), ('robust_etg', highlum_masks.robust_etg), ('robust_ltg', highlum_masks.robust_ltg)]:
    for variable in ('MP_LTG', 'MP_EdgeOn', 'MAG_AUTO_R', 'FLUX_RADIUS_R', 'Separation'):
        summary_rows.append(describe_values('highlum', class_name, variable, highlum[variable][mask]))
summary_table = Table(rows=[tuple(asdict(row).values()) for row in summary_rows], names=SUMMARY_HEADERS)
threshold_result_rows = threshold_rows('highlum', highlum['MP_LTG'], highlum['FLAG_LTG'])
threshold_table = Table(rows=[tuple(asdict(row).values()) for row in threshold_result_rows], names=THRESHOLD_HEADERS)
ltg_models = np.column_stack([highlum[f'P{i}_LTG'] for i in range(1, 6)])
edgeon_models = np.column_stack([highlum[f'P{i}_EdgeOn'] for i in range(1, 6)])
ltg_dispersion = model_dispersion(ltg_models)
edgeon_dispersion = model_dispersion(edgeon_models)

# Independent values measured from the local highlum file make this an integration regression check.
baseline = {
    ('robust_etg', 'MP_LTG'): (0.01198184, 0.00148256, 0.02721692),
    ('robust_etg', 'MP_EdgeOn'): (0.02750497, 0.00760955, 0.07180363),
    ('robust_ltg', 'MP_LTG'): (0.95182968, 0.96886268, 0.04962162),
    ('robust_ltg', 'MP_EdgeOn'): (0.13879124, 0.06322150, 0.18570282),
}
for row in summary_rows:
    key = (row.class_name, row.variable)
    if key in baseline:
        np.testing.assert_allclose((row.mean, row.median, row.std_ddof1), baseline[key], atol=1e-7, rtol=0)

# Persist the exact tables displayed in the notebook, then verify their public column contracts.
flag_table.write(highlum_tables / 'flag_counts.csv', format='ascii.csv', overwrite=True)
summary_table.write(highlum_tables / 'summary_statistics.csv', format='ascii.csv', overwrite=True)
threshold_table.write(highlum_tables / 'threshold_sensitivity.csv', format='ascii.csv', overwrite=True)
assert tuple(Table.read(highlum_tables / 'flag_counts.csv', format='ascii.csv').colnames) == FLAG_COUNT_HEADERS
assert tuple(Table.read(highlum_tables / 'summary_statistics.csv', format='ascii.csv').colnames) == SUMMARY_HEADERS
assert tuple(Table.read(highlum_tables / 'threshold_sensitivity.csv', format='ascii.csv').colnames) == THRESHOLD_HEADERS
display(flag_table)
display(summary_table[["class", "variable", "n_valid", "mean", "median", "std_ddof1", "p25", "p75"]])
display(threshold_table)


### Reading the probability summaries

Robust ETGs should have `MP_LTG` concentrated near zero, while robust LTGs should concentrate near one; the verified medians reflect exactly that operational definition. `MP_EdgeOn` is not another morphology class: it represents viewing orientation and is especially relevant for disks whose spiral structure may be hidden in projection.

The P1–P5 standard deviation and range measure disagreement among the five trained networks. They are useful uncertainty diagnostics, but they do not include every source of uncertainty—such as image depth, selection effects, domain shift, or incorrect labels used during training.

## 7. Magnitude–size relation

The next four views separate coverage, density, filtering, and robust-class comparison. `MAG_AUTO_R` is an apparent magnitude: smaller numerical values are brighter. `FLUX_RADIUS_R` is an observed angular-image scale reported in pixels; without a redshift-dependent angular-diameter conversion it must not be interpreted as physical galaxy size. Scatter rendering uses a deterministic sample only when a catalogue exceeds `PLOT_SAMPLE_SIZE`; calculations above always use every valid row.

In [ ]:
# Figure constructors are pure; this orchestration cell owns deterministic sampling, saving, and display.
import matplotlib.pyplot as plt

from src.galaxy_analysis.plotting import (
    plot_edgeon_vs_ltg_probability,
    plot_flag_counts,
    plot_magnitude_radius_density,
    plot_magnitude_radius_scatter,
    plot_model_dispersion_by_class,
    plot_probability_by_class,
    plot_probability_vs_magnitude,
    plot_sky_distribution,
)

highlum_figures = OUTPUT_ROOT / 'highlum' / 'figures'
highlum_figures.mkdir(parents=True, exist_ok=True)
# Sample only point rendering. highlum is smaller than the cap, so every row is retained in this run.
highlum_plot_indices = random_indices(len(highlum['FLAG_LTG']), min(PLOT_SAMPLE_SIZE, len(highlum['FLAG_LTG'])), SEED)
highlum_plot_masks = robust_masks(highlum['FLAG_LTG'][highlum_plot_indices])

def save_and_show(figure, filename, directory=highlum_figures):
    """Save one stable 150-dpi PNG, display that same figure inline, and release its memory.

    Saving before closing guarantees the local report artifact matches the notebook view; closing matters when
    dozens of large figures are generated in one kernel.
    """
    path = directory / filename
    figure.savefig(path, dpi=150, bbox_inches='tight')
    display(figure)
    plt.close(figure)
    return path


### Figure 7.1 — Apparent magnitude versus flux radius: all valid rows

1. **Question:** What region of observed brightness–size space is occupied by the matched `highlum` catalogue before morphology filtering?
2. **Variables and encoding:** The horizontal axis is `MAG_AUTO_R` in magnitudes and is inverted so brighter objects lie to the left; the vertical axis is `FLUX_RADIUS_R` in pixels. Every plotted point is a row with two finite values, shown in grey.
3. **Why this visualization:** A scatter plot exposes the support, boundaries, sparse outliers, and possible measurement sequences that a one-dimensional summary would hide.
4. **How to read it:** Moving left means brighter apparent flux; moving upward means larger observed angular extent on the detector. Dense overlap appears darker, but point darkness is not a normalized density estimate.
5. **What to examine:** Look for implausible radii, sharp survey-dependent boundaries, magnitude-dependent radius scatter, and isolated measurements that could dominate axis limits.
6. **Limitations:** Apparent magnitude mixes luminosity and distance, while pixel radius is not physical radius. Crowding, seeing, detection thresholds, and catalogue selection can shape the relation; it cannot establish a causal size–luminosity law.

In [ ]:
# Coverage view: plot all finite magnitude/radius pairs without applying morphology selection.
figure = plot_magnitude_radius_scatter(
    highlum['MAG_AUTO_R'][highlum_plot_indices],
    highlum['FLUX_RADIUS_R'][highlum_plot_indices],
    None,
    'highlum',
)
save_and_show(figure, 'magnitude_vs_flux_radius_all.png')


### Figure 7.2 — Provisional radius cut

1. **Question:** Does restricting `FLUX_RADIUS_R < 50` reveal the central relation more clearly, and how many otherwise valid rows does it remove?
2. **Variables and encoding:** Axes and magnitude direction match Figure 7.1; the only new filter is the strict pixel-radius cut. The annotation reports its removal count from the full valid magnitude–radius sample.
3. **Why this visualization:** A matched scatter plot makes the visual effect of the proposed cut directly comparable to the uncut view.
4. **How to read it:** Any change is due to removing points at or above 50 pixels, not a changed magnitude definition. If zero rows are removed, the two point clouds should coincide.
5. **What to examine:** Check whether removed rows are isolated failures or a coherent bright/extended population that deserves scientific retention.
6. **Limitations:** The value 50 is provisional and has no demonstrated physical meaning here. It may remove zero `highlum` rows or may discard genuine extended galaxies; it is never used for the primary statistics or robust-class definition.

In [ ]:
# Audit the provisional radius cut on the full sample before applying it only to this diagnostic figure.
valid_magnitude_radius = np.isfinite(highlum['MAG_AUTO_R']) & np.isfinite(highlum['FLUX_RADIUS_R'])
radius_cut_removed = int(np.count_nonzero(valid_magnitude_radius & (highlum['FLUX_RADIUS_R'] >= FLUX_RADIUS_CUT)))
figure = plot_magnitude_radius_scatter(
    highlum['MAG_AUTO_R'][highlum_plot_indices],
    highlum['FLUX_RADIUS_R'][highlum_plot_indices],
    None,
    'highlum',
    flux_radius_max=FLUX_RADIUS_CUT,
)
figure.axes[0].text(0.02, 0.98, f'Full valid sample: {radius_cut_removed:,} removed', transform=figure.axes[0].transAxes, va='top')
save_and_show(figure, 'magnitude_vs_flux_radius_cut50.png')


### Figure 7.3 — Magnitude–radius density

1. **Question:** Where is the complete valid sample concentrated in magnitude–radius space?
2. **Variables and encoding:** The axes retain the Figure 7.1 definitions. Hexagon color encodes the logarithmic number of valid catalogue rows in a fixed spatial bin; all finite rows are used.
3. **Why this visualization:** Hexagonal binning avoids severe overplotting and reveals both a dense locus and lower-occupancy structure without sampling.
4. **How to read it:** Brighter objects are left, larger pixel radii are up, and lighter/high-valued colorbar regions contain more measurements per hexagon. Empty background is not interpolated.
5. **What to examine:** Look for multiple loci, abrupt edges, low-density wings, and whether scatter broadens toward fainter magnitudes.
6. **Limitations:** Logarithmic bin counts depend on bin size and sample selection. Density does not identify intrinsic subpopulations, and seeing or surface-brightness limits can create apparent trends.

In [ ]:
# Hexbin uses all valid rows so the dense magnitude–radius locus is not shaped by scatter sampling.
figure = plot_magnitude_radius_density(highlum['MAG_AUTO_R'], highlum['FLUX_RADIUS_R'], 'highlum')
save_and_show(figure, 'magnitude_vs_flux_radius_density.png')


### Figure 7.4 — Robust morphology classes in magnitude–radius space

1. **Question:** Do catalogue-defined robust ETGs (flag 4) and robust LTGs (flag 5) occupy visibly different observed brightness–size regions?
2. **Variables and encoding:** Red marks robust ETGs and blue marks robust LTGs; labels include valid plotted `N`. Flags 0–3 are deliberately excluded rather than relabelled by parity. Both catalogue versions use the same fixed magnitude `(21.6, 18.0)` and radius `(2.5, 22.0)` limits, including the inverted magnitude direction, so their geometry is directly comparable.
3. **Why this visualization:** A transparent, rasterized scatter plot preserves object-level overlap while comparing two explicitly selected classes.
4. **How to read it:** Separation between colors suggests differing observed distributions; overlap means the two catalogue classes share that region. Darker areas can partly reflect the much larger ETG sample.
5. **What to examine:** Compare envelopes, faint limits, extreme radii, and whether the small LTG sample follows the same locus.
6. **Limitations:** Unequal class sizes affect visual prominence. Classification, magnitude, and radius all originate in related imaging data, so correlations may reflect shared measurement quality; observed separation is not proof of a physical mechanism.

In [ ]:
# Compare only flag-4 and flag-5 objects; non-robust flags are intentionally absent from class colors.
figure = plot_magnitude_radius_scatter(
    highlum['MAG_AUTO_R'][highlum_plot_indices],
    highlum['FLUX_RADIUS_R'][highlum_plot_indices],
    highlum_plot_masks,
    'highlum',
)
# Apply the registered common limits after construction; this preserves one reusable plotting function.
figure.axes[0].set_xlim(COMPARISON_MAGNITUDE_LIMITS)
figure.axes[0].set_ylim(COMPARISON_RADIUS_LIMITS)
save_and_show(figure, 'magnitude_size_robust_classes.png')


## 8. Probability, faintness, and edge-on orientation

Probabilities are model outputs on `[0, 1]`, not observed frequencies guaranteed to be calibrated. The plots below retain the catalogue's robust flag selection where class comparisons are made and use normalized distributions when class sizes differ.

### Figure 8.1 — LTG probability by robust class

1. **Question:** How strongly does aggregate `MP_LTG` separate robust flag-4 and flag-5 objects?
2. **Variables and encoding:** The horizontal axis is `MP_LTG` from 0 to 1; the vertical axis is normalized density. Red is robust ETG, blue is robust LTG, and both use identical bins with valid `N` in the legend.
3. **Why this visualization:** Overlaid normalized histograms compare distribution shape despite unequal counts.
4. **How to read it:** Mass near 0 denotes low LTG output and mass near 1 denotes high LTG output; overlapping density identifies ambiguous or boundary regions.
5. **What to examine:** Inspect peak location, tails toward the opposite class, and any secondary modes.
6. **Limitations:** Peaks near 0 or 1 indicate confident model outputs, not guaranteed truth or calibration. Flags and probabilities are related catalogue products, so their agreement is not an independent validation.

In [ ]:
# Normalized bins compare probability shape despite the much larger robust-ETG sample.
figure = plot_probability_by_class(highlum['MP_LTG'], highlum_masks, 'MP_LTG', 'highlum')
save_and_show(figure, 'mp_ltg_by_robust_class.png')


### Figure 8.2 — Edge-on probability by robust class

1. **Question:** How is the aggregate edge-on output distributed within the two robust morphology classes?
2. **Variables and encoding:** The horizontal axis is `MP_EdgeOn` on `[0, 1]`; normalized density, bins, class colors, and valid-count labels follow Figure 8.1.
3. **Why this visualization:** Matching histograms show whether orientation-related outputs differ in location, spread, or tails between classes.
4. **How to read it:** Values to the right mean the ensemble assigns higher edge-on probability; overlap means the probability alone does not distinguish catalogue class.
5. **What to examine:** Look for a high-probability LTG tail and for ETGs with unexpectedly large orientation scores that merit image inspection.
6. **Limitations:** Edge-on is an orientation prediction, not a third morphology class. Projection, dust, resolution, and training-set biases can alter this output, and class-size imbalance remains present despite density normalization.

In [ ]:
# Treat edge-on probability as an orientation output, not as another morphology class.
figure = plot_probability_by_class(highlum['MP_EdgeOn'], highlum_masks, 'MP_EdgeOn', 'highlum')
save_and_show(figure, 'mp_edgeon_by_robust_class.png')


### Figure 8.3 — Disagreement among the five LTG models

1. **Question:** Within each robust class, how much do P1–P5 disagree for an individual object?
2. **Variables and encoding:** The vertical axis is the per-row sample standard deviation (`ddof=1`) of P1–P5 LTG probabilities. Boxes show median and interquartile range; red/blue retain the robust-class palette and labels report `N`.
3. **Why this visualization:** A box plot summarizes central disagreement and the middle 50% without allowing extreme values to compress the main comparison.
4. **How to read it:** Higher boxes mean greater model-to-model dispersion; a wider interquartile range means less uniform agreement across objects in that class.
5. **What to examine:** Compare medians and IQRs and identify whether one class contains systematically less stable predictions.
6. **Limitations:** Five correlated networks are not five independent measurements. This is one component of epistemic uncertainty and omits calibration error, data shift, label error, and observational noise.

In [ ]:
# Visualize P1–P5 disagreement; this is a model-dispersion diagnostic, not complete uncertainty.
figure = plot_model_dispersion_by_class(ltg_dispersion['std_ddof1'], highlum_masks, 'highlum')
save_and_show(figure, 'model_dispersion_ltg.png')


### Figure 8.4 — LTG probability versus apparent faintness

1. **Question:** Does the distribution of `MP_LTG` change toward fainter apparent magnitudes?
2. **Variables and encoding:** `MAG_AUTO_R` increases toward fainter sources on the horizontal axis; `MP_LTG` occupies `[0, 1]` vertically. Hexagon color is logarithmic occupancy for every finite, in-range pair.
3. **Why this visualization:** Hexbin density uses all valid rows while avoiding overplotting in a large, highly concentrated probability sample.
4. **How to read it:** Moving right means fainter apparent flux; moving up means stronger LTG output. Vertical broadening toward the right would indicate more diverse model outputs at fainter magnitudes.
5. **What to examine:** Look for changes in concentration near 0/1, a growing intermediate-probability population, and magnitude-dependent boundaries.
6. **Limitations:** Increasing ambiguity toward faint sources may reflect lower signal-to-noise, resolution, population mix, or selection. The plot is observational and must not be described as evidence that faintness causes classification uncertainty.

In [ ]:
# Use all valid rows to inspect whether probability ambiguity changes toward fainter apparent magnitudes.
figure = plot_probability_vs_magnitude(highlum['MAG_AUTO_R'], highlum['MP_LTG'], 'highlum')
save_and_show(figure, 'ltg_probability_vs_magnitude.png')


### Figure 8.5 — Edge-on and LTG probabilities jointly

1. **Question:** How do orientation and late-type model outputs co-vary across the matched sample?
2. **Variables and encoding:** `MP_LTG` is horizontal and `MP_EdgeOn` vertical, both fixed to `[0, 1]`; logarithmic hexagon occupancy includes every valid pair.
3. **Why this visualization:** A two-dimensional density plot reveals concentrations and curved or branched structure that separate histograms cannot show.
4. **How to read it:** Upper-right bins have high outputs for both properties; lower-right bins have high LTG but low edge-on output; color records how many rows occupy each bin.
5. **What to examine:** Inspect whether high edge-on values occur mainly after LTG probability rises and whether intermediate LTG outputs show elevated orientation uncertainty.
6. **Limitations:** Orientation can obscure spiral structure and correlate classification uncertainties, but correlation is not a causal estimate. Both axes come from related models and may share training and image-quality biases.

In [ ]:
# Joint density reveals orientation/morphology structure that separate one-dimensional histograms cannot.
figure = plot_edgeon_vs_ltg_probability(highlum['MP_LTG'], highlum['MP_EdgeOn'], 'highlum')
save_and_show(figure, 'edgeon_vs_ltg_probability.png')


## 9. Sky distribution and catalogue composition

These maps show observed catalogue coordinates directly; they do not interpolate across unobserved areas or imply that every gap has an astrophysical origin.

### Figure 9.1 — All valid sky positions

1. **Question:** What angular footprint and obvious coverage structure does the matched `highlum` catalogue have?
2. **Variables and encoding:** Right ascension `RA_2` and declination `DEC_2` are in degrees. Grey rasterized points represent valid coordinate pairs from the deterministic rendering sample.
3. **Why this visualization:** A direct scatter map preserves holes and edges instead of smoothing across places with no catalogue entries.
4. **How to read it:** Dense patches contain more matched objects per displayed sky area; blank regions mean no plotted catalogue rows, not necessarily an absence of galaxies.
5. **What to examine:** Check boundaries, stripes, disconnected fields, and small holes that might align with observing or masking patterns.
6. **Limitations:** Holes may result from survey masks around saturated stars, footprint geometry, depth, matching, or real density variation. Without an official mask, no cause should be assigned to an individual gap and a flat RA–DEC view does not preserve area globally.

In [ ]:
# Plot observed coordinates directly so footprint and mask holes remain empty rather than interpolated.
figure = plot_sky_distribution(highlum['RA_2'][highlum_plot_indices], highlum['DEC_2'][highlum_plot_indices], None, 'highlum')
save_and_show(figure, 'sky_distribution_all.png')


### Figure 9.2 — Robust classes across the footprint

1. **Question:** Are robust ETG and LTG catalogue selections distributed similarly across the observed footprint?
2. **Variables and encoding:** Coordinates and projection match Figure 9.1. Red points are flag-4 ETGs and blue points flag-5 LTGs, with valid plotted `N`; flags 0–3 are excluded.
3. **Why this visualization:** An identically projected class overlay can reveal field-dependent composition or classification anomalies while retaining footprint gaps.
4. **How to read it:** Local color balance suggests catalogue composition in a displayed area, but the visually dominant red class is also globally much larger.
5. **What to examine:** Look for regions containing one class disproportionately, class-specific edges, and patterns aligned with coverage rather than celestial structure.
6. **Limitations:** Unequal counts, selection functions, spatially varying depth, seeing, and saturated-star masks can all produce apparent differences. This map is descriptive and cannot demonstrate environmental causation.

In [ ]:
# Overlay robust classes on the identical sky projection to expose spatial selection differences.
figure = plot_sky_distribution(
    highlum['RA_2'][highlum_plot_indices],
    highlum['DEC_2'][highlum_plot_indices],
    highlum_plot_masks,
    'highlum',
)
save_and_show(figure, 'sky_distribution_robust_classes.png')


### Figure 9.3 — Complete `FLAG_LTG` composition

1. **Question:** How many matched objects receive each flag, including classes excluded from the robust analysis?
2. **Variables and encoding:** Bars cover flags 0–5; height is raw galaxy count and text reports count plus percentage of all rows. Flags 4 and 5 use the ETG/LTG palette, while 0–3 remain neutral because they are non-robust.
3. **Why this visualization:** A bar chart represents a discrete categorical variable without implying continuity between flag values.
4. **How to read it:** Taller bars mean more catalogue rows with that exact flag. Percentages use the full matched catalogue denominator.
5. **What to examine:** Compare robust versus non-robust mass, the imbalance between flags 4 and 5, and whether any expected flag is absent.
6. **Limitations:** Raw counts reflect population, source selection, matching, image quality, and classifier rules together. They are not intrinsic cosmic morphology fractions and do not make flags 0–3 safe to fold into robust classes by even/odd parity.

In [ ]:
# Show all flags so the non-robust population excluded from primary analysis remains quantitatively visible.
figure = plot_flag_counts(highlum['FLAG_LTG'], 'highlum')
save_and_show(figure, 'flag_ltg_counts.png')


## 10. `highlum` versus `highdens`: a controlled catalogue comparison

The names `highlum` and `highdens` are treated as catalogue aliases, not as complete physical definitions. Until the source-selection functions, completeness, overlap, and weighting are documented, differences are descriptive associations between two matched products—not estimates of an environmental causal effect.

The `highdens` file is processed only after the verified `highlum` baseline has passed. It uses the identical 21-column request, validity domains, one-arcsecond rule, flag-4/flag-5 selection, statistics, plot constructors, axis meanings, and stable output names. This avoids reading all 191 FITS columns and makes any contrast traceable to catalogue contents rather than different analysis code.

In [ ]:
# Scale to highdens only after every highlum regression assertion has passed above.
highdens = read_columns(CATALOG_PATHS['highdens'], CORE_COLUMNS)
# This assertion is the notebook-level memory contract: only the named 21 columns may be copied.
assert tuple(highdens) == CORE_COLUMNS
highdens_quality_masks = {
    'RA_2 in [0, 360) deg': valid_value_mask(highdens['RA_2'], 'ra_deg'),
    'DEC_2 in [-90, 90] deg': valid_value_mask(highdens['DEC_2'], 'dec_deg'),
    'FLUX_RADIUS_R positive': valid_value_mask(highdens['FLUX_RADIUS_R'], 'positive'),
    'MP_LTG in [0, 1]': valid_value_mask(highdens['MP_LTG'], 'probability'),
    'MP_EdgeOn in [0, 1]': valid_value_mask(highdens['MP_EdgeOn'], 'probability'),
    'Separation in [0, 1] arcsec': valid_value_mask(highdens['Separation'], 'nonnegative') & (highdens['Separation'] <= MAX_SEPARATION_ARCSEC),
}
# Reuse identical validity rules so differences cannot be caused by catalogue-specific cleaning code.
highdens_audit_rows = [audit_filter('highdens quality', rule, mask) for rule, mask in highdens_quality_masks.items()]
highdens_audit_table = Table(
    rows=[('highdens', *tuple(asdict(row).values())) for row in highdens_audit_rows],
    names=FILTER_AUDIT_HEADERS,
)
highdens_separation_check = validate_separation(highdens['Separation'], MAX_SEPARATION_ARCSEC)
assert highdens_separation_check.is_valid, f'{highdens_separation_check.invalid_count} highdens matches exceed quality limits'
highdens_masks = robust_masks(highdens['FLAG_LTG'])
highdens_flag_values, highdens_counts_array = np.unique(highdens['FLAG_LTG'], return_counts=True)
highdens_flag_counts = dict(zip(highdens_flag_values.astype(int), highdens_counts_array.astype(int)))
# Record a second exact file/selection baseline before any comparison is interpreted.
assert len(highdens['FLAG_LTG']) == 905_291
assert highdens_flag_counts == {0: 160930, 1: 194143, 2: 66841, 3: 51152, 4: 369660, 5: 62565}
assert np.array_equal(highdens_masks.robust_etg, highdens['FLAG_LTG'] == 4)
assert np.array_equal(highdens_masks.robust_ltg, highdens['FLAG_LTG'] == 5)
highdens_audit_table


In [ ]:
# Generate the same products as highlum so both catalogues remain directly comparable.
from src.galaxy_analysis.statistics import compare_catalog_summaries

# 1. Prepare separate output folders for tables and Matplotlib figures.
highdens_output = OUTPUT_ROOT / 'highdens'
highdens_tables = highdens_output / 'tables'
highdens_figures = highdens_output / 'figures'

highdens_tables.mkdir(parents=True, exist_ok=True)
highdens_figures.mkdir(parents=True, exist_ok=True)

# 2. Summarize every loaded column.
# Numerical columns receive descriptive statistics.
# Identifier columns keep only completeness counts.
highdens_quality_rows = []

for name, values in highdens.items():
    array = np.asarray(values)
    is_numeric = np.issubdtype(array.dtype, np.number)

    if is_numeric:
        numeric = array.astype(float)
        finite_mask = np.isfinite(numeric)
        valid_values = numeric[finite_mask]

        if valid_values.size:
            numeric_statistics = (
                float(valid_values.min()),
                float(valid_values.max()),
                float(valid_values.mean()),
                float(np.median(valid_values)),
            )
        else:
            numeric_statistics = (None, None, None, None)

        n_finite = int(finite_mask.sum())
        n_missing = int((~finite_mask).sum())
    else:
        numeric_statistics = (None, None, None, None)
        n_finite = int(array.size)
        n_missing = 0

    quality_row = (
        'highdens',
        name,
        str(array.dtype),
        schemas['highdens'].column_units.get(name),
        len(array),
        n_finite,
        n_missing,
        0,  # Domain violations were already audited in the previous cell.
        *numeric_statistics,
    )
    highdens_quality_rows.append(quality_row)

highdens_quality_table = Table(
    rows=highdens_quality_rows,
    names=QUALITY_HEADERS,
)

# 3. Build flag, descriptive-statistics, and threshold-sensitivity tables.
highdens_flag_rows = flag_count_rows('highdens', highdens['FLAG_LTG'])
highdens_flag_table = Table(
    rows=[tuple(asdict(row).values()) for row in highdens_flag_rows],
    names=FLAG_COUNT_HEADERS,
)

highdens_samples = {
    'all_valid': np.ones(len(highdens['FLAG_LTG']), dtype=bool),
    'robust_etg': highdens_masks.robust_etg,
    'robust_ltg': highdens_masks.robust_ltg,
}
highdens_summary_variables = (
    'MP_LTG',
    'MP_EdgeOn',
    'MAG_AUTO_R',
    'FLUX_RADIUS_R',
    'Separation',
)

highdens_summary_rows = []

for class_name, sample_mask in highdens_samples.items():
    for variable in highdens_summary_variables:
        summary_row = describe_values(
            'highdens',
            class_name,
            variable,
            highdens[variable][sample_mask],
        )
        highdens_summary_rows.append(summary_row)

highdens_summary_table = Table(
    rows=[tuple(asdict(row).values()) for row in highdens_summary_rows],
    names=SUMMARY_HEADERS,
)

highdens_threshold_rows = threshold_rows(
    'highdens',
    highdens['MP_LTG'],
    highdens['FLAG_LTG'],
)
highdens_threshold_table = Table(
    rows=[tuple(asdict(row).values()) for row in highdens_threshold_rows],
    names=THRESHOLD_HEADERS,
)

# 4. Measure disagreement among the five catalogue model outputs.
# Matplotlib visualizes these summaries in the following cell.
# Delete the temporary two-dimensional matrices after calculating
# their row-wise summaries.
highdens_ltg_models = np.column_stack([highdens[f'P{i}_LTG'] for i in range(1, 6)])
highdens_edgeon_models = np.column_stack(
    [highdens[f'P{i}_EdgeOn'] for i in range(1, 6)]
)

highdens_ltg_dispersion = model_dispersion(highdens_ltg_models)
highdens_edgeon_dispersion = model_dispersion(highdens_edgeon_models)

del highdens_ltg_models, highdens_edgeon_models

# 5. Save every table and immediately verify its serialized column names.
highdens_table_outputs = {
    'catalog_quality.csv': (highdens_quality_table, QUALITY_HEADERS),
    'flag_counts.csv': (highdens_flag_table, FLAG_COUNT_HEADERS),
    'summary_statistics.csv': (highdens_summary_table, SUMMARY_HEADERS),
    'threshold_sensitivity.csv': (highdens_threshold_table, THRESHOLD_HEADERS),
    'filter_audit.csv': (highdens_audit_table, FILTER_AUDIT_HEADERS),
}

for filename, (table, expected_headers) in highdens_table_outputs.items():
    table_path = highdens_tables / filename
    table.write(table_path, format='ascii.csv', overwrite=True)

    saved_headers = tuple(Table.read(table_path, format='ascii.csv').colnames)
    assert saved_headers == expected_headers

# 6. Record file-level quality facts and explicit metadata limitations.
highdens_duplicate_morphology_ids = len(highdens['COADD_OBJECT_ID']) - len(
    np.unique(highdens['COADD_OBJECT_ID'])
)
highdens_duplicate_environment_ids = len(highdens['object_id']) - len(
    np.unique(highdens['object_id'])
)

highdens_quality_report = {
    'catalog': 'highdens',
    'rows': schemas['highdens'].row_count,
    'columns_available': schemas['highdens'].column_count,
    'columns_loaded': list(CORE_COLUMNS),
    'separation_unit': schemas['highdens'].column_units['Separation'],
    'separation_invalid_count': highdens_separation_check.invalid_count,
    'separation_maximum_valid_arcsec': highdens_separation_check.maximum_valid,
    'duplicate_morphology_ids': int(highdens_duplicate_morphology_ids),
    'duplicate_environment_ids': int(highdens_duplicate_environment_ids),
    'warnings': [
        'RA/DEC units are not declared in the FITS header; '
        'catalogue convention is degrees'
    ],
}
quality_report_path = highdens_output / 'quality_report.json'
quality_report_path.write_text(
    json.dumps(highdens_quality_report, indent=2),
    encoding='utf-8',
)

# 7. Display focused review tables.
# Complete versions remain available in the saved CSV files.
summary_display_columns = (
    'class',
    'variable',
    'n_valid',
    'mean',
    'median',
    'std_ddof1',
    'p25',
    'p75',
)

display(highdens_flag_table)
display(highdens_summary_table[list(summary_display_columns)])
display(highdens_threshold_table)


### Repeated diagnostics for `highdens`

All twelve per-catalogue figures are regenerated for `highdens` with the stable filenames from Sections 7–9. Their six-part reading guides apply unchanged: magnitude direction, angular-versus-physical size, probability caveats, inter-model dispersion, orientation ambiguity, footprint masks, and flag-selection limitations do not depend on the alias. Numerical statistics and density plots use all valid rows; only object-level scatter and sky rendering are deterministically capped at 100,000 points. The radius `<50` view remains provisional and reports its own removal count.

In [ ]:
# Matplotlib receives at most 100,000 rows for point-based plots.
# Histograms and hexbin density plots still use every valid catalogue row.
highdens_plot_indices = random_indices(
    population_size=len(highdens['FLAG_LTG']),
    sample_size=PLOT_SAMPLE_SIZE,
    seed=SEED,
)

# Give the sampled arrays descriptive names so the plotting calls read like a recipe.
sampled_highdens_magnitude = highdens['MAG_AUTO_R'][highdens_plot_indices]
sampled_highdens_radius = highdens['FLUX_RADIUS_R'][highdens_plot_indices]
sampled_highdens_ra = highdens['RA_2'][highdens_plot_indices]
sampled_highdens_dec = highdens['DEC_2'][highdens_plot_indices]
sampled_highdens_flags = highdens['FLAG_LTG'][highdens_plot_indices]
sampled_highdens_masks = robust_masks(sampled_highdens_flags)

# Count the provisional radius cut on the complete catalogue,
# not on the smaller plotting sample.
highdens_valid_magnitude_radius = (
    np.isfinite(highdens['MAG_AUTO_R'])
    & np.isfinite(highdens['FLUX_RADIUS_R'])
)
highdens_radius_cut_removed = int(
    np.count_nonzero(
        highdens_valid_magnitude_radius
        & (highdens['FLUX_RADIUS_R'] >= FLUX_RADIUS_CUT)
    )
)

# Figure 1: all sampled magnitude–radius pairs, before morphology filtering.
figure = plot_magnitude_radius_scatter(
    magnitude=sampled_highdens_magnitude,
    radius=sampled_highdens_radius,
    masks=None,
    catalog_alias='highdens',
)
save_and_show(
    figure,
    'magnitude_vs_flux_radius_all.png',
    highdens_figures,
)

# Figure 2: the same relation with the advisor's provisional radius cut.
figure = plot_magnitude_radius_scatter(
    magnitude=sampled_highdens_magnitude,
    radius=sampled_highdens_radius,
    masks=None,
    catalog_alias='highdens',
    flux_radius_max=FLUX_RADIUS_CUT,
)
figure.axes[0].text(
    0.02,
    0.98,
    f'Full valid sample: {highdens_radius_cut_removed:,} removed',
    transform=figure.axes[0].transAxes,
    verticalalignment='top',
)
save_and_show(
    figure,
    'magnitude_vs_flux_radius_cut50.png',
    highdens_figures,
)

# Figure 3: Matplotlib hexbin density from every finite magnitude–radius pair.
figure = plot_magnitude_radius_density(
    magnitude=highdens['MAG_AUTO_R'],
    radius=highdens['FLUX_RADIUS_R'],
    catalog_alias='highdens',
)
save_and_show(
    figure,
    'magnitude_vs_flux_radius_density.png',
    highdens_figures,
)

# Figure 4: robust ETGs and LTGs on limits shared with the highlum figure.
figure = plot_magnitude_radius_scatter(
    magnitude=sampled_highdens_magnitude,
    radius=sampled_highdens_radius,
    masks=sampled_highdens_masks,
    catalog_alias='highdens',
)
figure.axes[0].set_xlim(COMPARISON_MAGNITUDE_LIMITS)
figure.axes[0].set_ylim(COMPARISON_RADIUS_LIMITS)
save_and_show(
    figure,
    'magnitude_size_robust_classes.png',
    highdens_figures,
)

# Figure 5: normalized LTG-probability distributions by robust class.
figure = plot_probability_by_class(
    probability=highdens['MP_LTG'],
    masks=highdens_masks,
    probability_name='MP_LTG',
    catalog_alias='highdens',
)
save_and_show(figure, 'mp_ltg_by_robust_class.png', highdens_figures)

# Figure 6: normalized edge-on-probability distributions by robust class.
figure = plot_probability_by_class(
    probability=highdens['MP_EdgeOn'],
    masks=highdens_masks,
    probability_name='MP_EdgeOn',
    catalog_alias='highdens',
)
save_and_show(figure, 'mp_edgeon_by_robust_class.png', highdens_figures)

# Figure 7: P1–P5 LTG model disagreement summarized by robust class.
figure = plot_model_dispersion_by_class(
    dispersion=highdens_ltg_dispersion['std_ddof1'],
    masks=highdens_masks,
    catalog_alias='highdens',
)
save_and_show(figure, 'model_dispersion_ltg.png', highdens_figures)

# Figure 8: LTG probability as apparent sources become fainter.
figure = plot_probability_vs_magnitude(
    magnitude=highdens['MAG_AUTO_R'],
    probability=highdens['MP_LTG'],
    catalog_alias='highdens',
)
save_and_show(figure, 'ltg_probability_vs_magnitude.png', highdens_figures)

# Figure 9: joint Matplotlib density of LTG and edge-on probabilities.
figure = plot_edgeon_vs_ltg_probability(
    ltg_probability=highdens['MP_LTG'],
    edgeon_probability=highdens['MP_EdgeOn'],
    catalog_alias='highdens',
)
save_and_show(figure, 'edgeon_vs_ltg_probability.png', highdens_figures)

# Figure 10: sampled sky footprint without filling survey or mask holes.
figure = plot_sky_distribution(
    ra_deg=sampled_highdens_ra,
    dec_deg=sampled_highdens_dec,
    masks=None,
    catalog_alias='highdens',
)
save_and_show(figure, 'sky_distribution_all.png', highdens_figures)

# Figure 11: the same sky projection split into robust ETG and LTG points.
figure = plot_sky_distribution(
    ra_deg=sampled_highdens_ra,
    dec_deg=sampled_highdens_dec,
    masks=sampled_highdens_masks,
    catalog_alias='highdens',
)
save_and_show(
    figure,
    'sky_distribution_robust_classes.png',
    highdens_figures,
)

# Figure 12: complete flag composition, including non-robust flags 0–3.
figure = plot_flag_counts(
    flags=highdens['FLAG_LTG'],
    catalog_alias='highdens',
)
save_and_show(figure, 'flag_ltg_counts.png', highdens_figures)


In [ ]:
# Define every comparison sign as highdens minus highlum and serialize the exact displayed rows.
comparison_output = OUTPUT_ROOT / 'comparison'
comparison_output.mkdir(parents=True, exist_ok=True)
comparison_rows = compare_catalog_summaries(summary_rows, highdens_summary_rows)
comparison_table = Table(
    rows=[tuple(asdict(row).values()) for row in comparison_rows],
    names=tuple(asdict(comparison_rows[0])),
)
comparison_table.write(comparison_output / 'summary_statistics_comparison.csv', format='ascii.csv', overwrite=True)
comparison_table


### Figure 10.1 — Robust-class fractions

1. **Question:** What fractions of each complete matched catalogue receive robust flag 4 or 5?
2. **Variables and encoding:** Bar height is the fraction of all rows in that catalogue; paired grey/green bars identify `highlum`/`highdens`, and error bars are 95% Wilson binomial intervals. ETG and LTG denominators include non-robust flags 0–3.
3. **Why this visualization:** Grouped bars expose absolute composition differences, while Wilson intervals state counting precision without a normal approximation.
4. **How to read it:** Compare aliases within one flag category and compare ETG/LTG imbalance within an alias; the vertical scale is a fraction from 0 to 1.
5. **What to examine:** Check whether the robust-total fraction and ETG/LTG balance change materially between catalogue products, not merely whether tiny intervals overlap.
6. **Limitations:** With very large samples, counting intervals can be narrow while selection-systematic uncertainty remains large. Fractions combine source selection, match success, image quality, and classification; they are not universal environmental morphology fractions.

In [ ]:
# Wilson intervals quantify counting precision; the surrounding text keeps selection uncertainty separate.
from src.galaxy_analysis.plotting import plot_parameter_comparison, plot_robust_fraction_comparison, plot_separation_comparison

figure = plot_robust_fraction_comparison(highlum_flag_rows, highdens_flag_rows)
save_and_show(figure, 'robust_class_fractions_comparison.png', comparison_output)


### Figure 10.2 — `MP_LTG` distributions by robust class and catalogue

1. **Question:** Within the same robust flag, do the two matched products have differently shaped LTG-probability distributions?
2. **Variables and encoding:** Panels separate flag-4 ETGs and flag-5 LTGs. The shared horizontal interval is `MP_LTG` on `[0,1]`; normalized outline histograms compare `highlum` and `highdens` without letting the larger catalogue dominate by count.
3. **Why this visualization:** Class-stratified normalized distributions control two major visual confounders—unequal catalogue size and unequal class size—while retaining tails and modes.
4. **How to read it:** A horizontal shift changes typical probability; different tail weight changes the frequency of less decisive outputs within the same robust flag. Shared limits make panels and aliases directly comparable.
5. **What to examine:** Compare peak sharpness near the operational class extreme, intermediate-probability tails, and the median differences recorded in the comparison CSV.
6. **Limitations:** Probability and flag share model provenance, the samples can overlap, and normalized density hides absolute counts. Differences may reflect brightness, redshift, image quality, or source rules; they do not isolate an environmental effect.

In [ ]:
# Normalize within robust class and enforce shared [0,1] probability limits across both aliases.
figure = plot_parameter_comparison(
    'MP_LTG',
    {'robust_etg': highlum['MP_LTG'][highlum_masks.robust_etg], 'robust_ltg': highlum['MP_LTG'][highlum_masks.robust_ltg]},
    {'robust_etg': highdens['MP_LTG'][highdens_masks.robust_etg], 'robust_ltg': highdens['MP_LTG'][highdens_masks.robust_ltg]},
)
save_and_show(figure, 'morphology_parameter_comparison.png', comparison_output)


### Figure 10.3 — Coordinate-match separation

1. **Question:** Do the two cross-match products have similar angular-separation quality within the accepted one-arcsecond limit?
2. **Variables and encoding:** The horizontal axis is `Separation` in arcseconds on a logarithmic scale; normalized outline histograms represent each complete matched product, and the dashed red line marks 1 arcsecond. An exact zero, if present, is retained in the first positive-width bin.
3. **Why this visualization:** Normalized histograms compare shape despite the 26-fold row-count difference. Logarithmic bins resolve the strong concentration at very small offsets while retaining the full view to the acceptance boundary.
4. **How to read it:** Density farther left indicates closer coordinate matches; a heavier right tail indicates more matches at larger angular offsets. Equal horizontal distances are multiplicative on this axis, and values are angular—not physical—distances.
5. **What to examine:** Compare modes, tails, and occupancy near the limit; consult `filter_audit.csv` and `quality_report.json` for invalid counts and maxima.
6. **Limitations:** Separation alone does not measure false-match probability, especially in crowded regions, and the plotted distributions are conditioned on the already-produced match catalogues. The line records the rule; it does not validate source identity.

In [ ]:
# Logarithmic separation bins resolve the ~1e-3 arcsec core while retaining the 1 arcsec rule.
figure = plot_separation_comparison(highlum['Separation'], highdens['Separation'], MAX_SEPARATION_ARCSEC)
save_and_show(figure, 'separation_distribution_comparison.png', comparison_output)


## 11. Reproducible random extraction from the parent morphology catalogue

Taking the first 100,000 rows is a contiguous preview, not a random sample: catalogue row order can encode sky position, observing batch, or an earlier sort. Here `np.random.default_rng(20260713)` selects unique integer positions without replacement, and the indices are sorted only after selection so FITS access can remain sequential and reproducible.

Only `sample_indices.npy` is saved. This preserves the exact selection while avoiding a second copy of catalogue data and lets a later analysis read only named columns at those positions. If the parent catalogue is smaller than the request, every row is selected and the recorded sample size is capped explicitly.

In [ ]:
# Save only reproducible row positions; catalogue values remain in their authoritative FITS source.
parent_schema = schemas['parent_morphology']
parent_sample_dir = OUTPUT_ROOT / 'parent-sample'
parent_sample_dir.mkdir(parents=True, exist_ok=True)
# Sampling without replacement avoids row-order bias and duplicate objects; sorting accelerates later FITS reads.
parent_sample_indices = random_indices(parent_schema.row_count, PLOT_SAMPLE_SIZE, SEED)
expected_parent_sample_size = min(parent_schema.row_count, PLOT_SAMPLE_SIZE)
assert len(parent_sample_indices) == expected_parent_sample_size
assert len(np.unique(parent_sample_indices)) == expected_parent_sample_size
assert np.all(parent_sample_indices[:-1] < parent_sample_indices[1:])
assert parent_sample_indices.min() >= 0
assert parent_sample_indices.max() < parent_schema.row_count
# NPY preserves integer dtype exactly; disabling pickle keeps the artifact data-only.
np.save(parent_sample_dir / 'sample_indices.npy', parent_sample_indices, allow_pickle=False)
parent_sample_summary = {
    'population_rows': parent_schema.row_count,
    'requested_rows': PLOT_SAMPLE_SIZE,
    'selected_rows': len(parent_sample_indices),
    'seed': SEED,
    'without_replacement': True,
    'saved_product': str((parent_sample_dir / 'sample_indices.npy').relative_to(PROJECT_ROOT)),
}
parent_sample_summary


## 12. Conclusions, limitations, and questions for the advisor

### Results supported by this execution

- The verified `highlum` match contains 34,768 rows: 24,871 robust ETGs (71.53% of the full match) and 890 robust LTGs (2.56%). The `highdens` match contains 905,291 rows: 369,660 robust ETGs (40.83%) and 62,565 robust LTGs (6.91%). These are catalogue-composition results, not population-universal morphology fractions.
- Both match products pass the explicit `Separation <= 1 arcsec` check with zero invalid rows and zero duplicate morphology or environmental identifiers. Median separation is 0.001286 arcsec for `highlum` and 0.001274 arcsec for `highdens`; rare maxima are 0.811 and 0.982 arcsec. The log-scale figure shows that most matches are orders of magnitude below the acceptance limit.
- Robust flags and aggregate probabilities align operationally: the `highlum` robust-ETG/robust-LTG `MP_LTG` medians are 0.00148/0.96886, while `highdens` medians are 0.02546/0.95236. Because flags and probabilities share model provenance, this is internal consistency rather than independent accuracy.
- Relative to `highlum`, the `highdens` robust ETG fraction is lower by 0.3070 and the robust LTG fraction higher by 0.0435. Robust objects in `highdens` also have brighter median `MAG_AUTO_R` by 0.859 mag (ETG) and 0.691 mag (LTG). These are observed sample differences recorded in `summary_statistics_comparison.csv`; no causal environmental claim follows.
- The provisional `FLUX_RADIUS_R < 50` view removes zero rows from both local match products. Primary statistics therefore remain uncut, and the value 50 is retained only as a documented diagnostic pending clarification.

### Working hypotheses to test later

- Different source-catalogue selection functions could explain much of the flag-composition contrast. A red-sequence, luminosity, density, redshift, or weighting rule may change the mixture before morphology is evaluated.
- The brighter `highdens` magnitude distribution and its different probability tails suggest image signal-to-noise, resolution, or population mix as potential confounders. A controlled comparison should stratify on magnitude and redshift before attributing residual structure to environment.
- Elevated edge-on probabilities and larger P1–P5 dispersion in subsets of robust LTGs may reflect orientation obscuring spiral structure. Image-level review and independent labels are needed to distinguish that hypothesis from calibration or training-domain effects.
- Footprint holes and spatial patterns may track saturated-star masks, depth, or tiling. They should be compared with an official survey mask before any large-scale-structure interpretation.

### Limitations

- Catalogue completeness, version provenance, `highlum`/`highdens` source definitions, overlap between samples, and selection weights have not yet been fully documented.
- No independent visual or spectroscopic truth labels are present. Probability calibration, flag agreement, and five-model dispersion cannot establish correctness by themselves.
- RA/DEC units are absent from the local FITS `TUNIT` metadata and are treated as degrees from convention and valid domain. Match separation validates proximity but is not a complete false-association model.
- `FLUX_RADIUS_R` is an image-plane pixel scale, not physical radius. Apparent magnitude, radius, redshift, morphology output, and image quality are correlated; simple plots do not isolate causal paths.
- The expected additional morphometric table from Fabrício is absent. Current radius and magnitude columns must not be presented as a complete morphometric feature set.
- Scatter/sky plots may use deterministic rendering samples; tables, counts, summaries, histograms, and hexbin density calculations use all valid matched rows.

### Questions for Arianna, with current evidence and safe provisional rules

1. **Does ‘50’ mean a hard radius filter or only a display limit?** Evidence: both match products have zero finite radii at or above 50. **Until confirmed:** retain both diagnostic versions and apply no radius cut to statistics.
2. **Is the local Vega-Ferrero parent catalogue the complete, current version?** Evidence: the file has 26,971,945 rows and 19 columns, with its path, size, modification time, HDU, and schema recorded in run metadata. **Until confirmed:** label every result with this local provenance and do not call it final.
3. **Did ‘John probability’ in the meeting mean `MP_EdgeOn`?** Evidence: the FITS products contain `MP_EdgeOn` and no `John` column, and the discussion concerned edge-on orientation. **Until confirmed:** use `MP_EdgeOn` and preserve this interpretation as a warning.
4. **Can RA/DEC be authoritatively confirmed as degrees despite missing `TUNIT`?** Evidence: values pass celestial coordinate domains and match the Topcat workflow. **Until confirmed:** treat them as degrees, validate ranges, and retain the metadata warning.
5. **Which morphometric parameters will Fabrício provide, and what are their units/definitions?** Evidence: current matches expose magnitude and radius fields but not the promised complete table. **Until received:** analyze only validated existing fields and avoid claiming complete morphometrics.
6. **Should the Topcat cross-matches be rebuilt in Python?** Evidence: both products have zero duplicate IDs and zero separations above 1 arcsec. **Until a concrete mismatch is found:** validate existing products and do not rebuild them.
7. **What are the precise physical selection rules and intended comparison for `highlum` and `highdens`?** Evidence: composition, brightness, and probability distributions differ substantially. **Until documented:** use the names only as file aliases and avoid environmental causal language.

In [ ]:
# Write an advisor-ready generated report from verified variables, while the detailed narrative remains visible above.
summary_report = f'''# Morphology catalogue analysis — meeting follow-up

## Results supported by this execution
- `highlum`: {len(highlum['FLAG_LTG']):,} rows; {int(highlum_masks.robust_etg.sum()):,} robust ETGs ({highlum_masks.robust_etg.mean():.4%}); {int(highlum_masks.robust_ltg.sum()):,} robust LTGs ({highlum_masks.robust_ltg.mean():.4%}).
- `highdens`: {len(highdens['FLAG_LTG']):,} rows; {int(highdens_masks.robust_etg.sum()):,} robust ETGs ({highdens_masks.robust_etg.mean():.4%}); {int(highdens_masks.robust_ltg.sum()):,} robust LTGs ({highdens_masks.robust_ltg.mean():.4%}).
- Both match products have zero rows outside the accepted 0–1 arcsec separation domain and zero duplicate identifiers.
- Quantitative distributions and `highdens - highlum` differences are in `highlum/tables/`, `highdens/tables/`, and `comparison/summary_statistics_comparison.csv`.
- Observed differences are descriptive catalogue associations, not environmental causal effects.

## Working hypotheses
Source selection, magnitude/redshift mixture, image quality, orientation, survey footprint, and model-domain shift may contribute to the observed differences. These require controlled follow-up analyses and independent labels.

## Limitations
Catalogue completeness and source-selection documentation are pending; RA/DEC lack FITS units; no independent truth labels or official mask are used; image-plane radius is not physical size; expected additional morphometrics are absent; correlated observables prevent causal interpretation.

## Questions for Arianna
1. Is 50 a radius filter or display limit? Provisional: diagnostic only.
2. Is the local 26,971,945-row parent catalogue complete/current? Provisional: report exact local provenance.
3. Does ‘John probability’ mean `MP_EdgeOn`? Provisional: use `MP_EdgeOn`.
4. Can RA/DEC units be confirmed as degrees? Provisional: degrees plus warning and range checks.
5. Which morphometrics and units will Fabrício provide? Provisional: use only current validated fields.
6. Should existing Topcat matches be rebuilt? Provisional: no, unless a documented defect appears.
7. What precisely do `highlum` and `highdens` select physically? Provisional: aliases only; no causal comparison.
'''
# The report is a reproducible ignored output, not a second manually maintained specification.
(comparison_output / 'summary.md').write_text(summary_report, encoding='utf-8')
print(summary_report)


### Reproducibility manifest

The final cell records software, Git revision, input file identity without expensive whole-file hashing, schemas, all execution parameters, generated product paths, quality warnings, and provisional decisions. The same run-level manifest is copied into each result area so a detached table or figure can be traced back to one execution.

In [ ]:
# Capture enough provenance to reproduce a run without hashing multi-gigabyte inputs every time.
import subprocess
from datetime import datetime, timezone

import astropy
import matplotlib

# The Git commit identifies code history; a failed lookup becomes null rather than aborting the science run.
git_result = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, check=False, capture_output=True, text=True
)
git_commit = git_result.stdout.strip() if git_result.returncode == 0 else None
# File size and modification time provide lightweight input identity alongside FITS schema facts.
input_manifest = {}
for alias, path in CATALOG_PATHS.items():
    stat = path.stat()
    schema = schemas[alias]
    input_manifest[alias] = {
        'path': str(path.relative_to(PROJECT_ROOT)),
        'size_bytes': stat.st_size,
        'modified_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
        'table_hdu': schema.hdu_index,
        'extname': schema.extname,
        'rows': schema.row_count,
        'columns': schema.column_count,
    }
# Enumerate outputs after generation so the manifest is also a completeness checklist.
generated_products = sorted(
    str(path.relative_to(OUTPUT_ROOT))
    for path in OUTPUT_ROOT.rglob('*')
    if path.is_file() and path.name != 'run_metadata.json'
)
generated_products.extend(['highlum/run_metadata.json', 'highdens/run_metadata.json', 'comparison/run_metadata.json'])
run_metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'software': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'astropy': astropy.__version__,
        'matplotlib': matplotlib.__version__,
    },
    'git_commit': git_commit,
    'inputs': input_manifest,
    'parameters': {
        'seed': SEED,
        'parent_sample_size_requested': PLOT_SAMPLE_SIZE,
        'plot_sample_size_maximum': PLOT_SAMPLE_SIZE,
        'maximum_separation_arcsec': MAX_SEPARATION_ARCSEC,
        'provisional_flux_radius_cut': FLUX_RADIUS_CUT,
        'robust_etg_rule': 'FLAG_LTG == 4',
        'robust_ltg_rule': 'FLAG_LTG == 5',
    },
    'generated_products': generated_products,
    'quality_warnings': [
        'RA_2 and DEC_2 lack FITS TUNIT metadata; degrees are provisionally assumed',
        'flags and probabilities are not independently labelled truth',
        'highlum and highdens remain file aliases pending source-selection documentation',
    ],
    'provisional_decisions': {
        'flux_radius_50': 'diagnostic only; not applied to primary statistics',
        'john_probability': 'interpreted as MP_EdgeOn pending confirmation',
        'crossmatches': 'validated in place; not rebuilt',
    },
}
# Copy one identical run manifest into each result area so detached products remain traceable.
for directory in (highlum_output, highdens_output, comparison_output):
    (directory / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print(json.dumps(run_metadata, indent=2))
